# BIO_0188 · S091 Gene Correlation

**Bioinformatics · tcga · L1 · public seed 1000**

This notebook is locked to the task selected on the RBioBench review website. Run **Setup**, inspect the task-aware data check, write your own R solution, and then run **Check output**. A reference solution is available at the end if you need it.

> Load 'counts.tsv' from the 'inputs' directory. Calculate the Spearman correlation coefficient between the expression profiles of the first two genes (first two rows) across all samples. Save the single numeric correlation value to 'outputs/correlation.txt'. Return the correlation value.

**Expected artifacts:** `outputs/correlation.txt`

## Input data · seed 1000

These are the actual files loaded by **Setup**. Large tables are previewed below.

### `inputs/clinical.tsv`

```text
barcode	vital_status	days_to_death	days_to_last_follow_up	gender
S01	Alive		969	male
S02	Dead	150	810	female
S03	Alive		297	male
S04	Dead	513	611	female
S05	Dead	509	488	male
S06	Dead	372	1324	female
S07	Dead	528	1340	male
S08	Dead	1588	630	female
```

### `inputs/counts.tsv`

```text
	S01	S02	S03	S04	S05	S06	S07	S08
TP53	796	439	685	778	101	403	361	64
BRCA1	479	169	547	447	133	227	687	246
EGFR	372	695	500	207	697	682	375	233
MYC	817	469	187	41	694	997	767	499
PIK3CA	146	472	731	727	425	708	909	205
PTEN	23	783	853	548	479	733	830	789
RB1	968	223	557	180	920	715	721	111
KRAS	258	164	292	95	1	321	833	16
AKT1	651	874	262	298	531	798	788	561
BRAF	121	198	929	299	390	327	674	233
CDH1	237	365	634	497	916	890	122	300
… [first 12 of 21 lines shown]
```

In [ ]:
# @title Setup
task_id <- "BIO_0188"
session_root <- "/content/rbiobench"
archive <- file.path(session_root, "practice.zip")
pack_dir <- file.path(session_root, "practice_pack")
workspace <- file.path(session_root, "workspace", task_id)
dir.create(session_root, recursive = TRUE, showWarnings = FALSE)

if (!file.exists(archive)) {
  download.file(
    "https://drive.usercontent.google.com/download?id=1e3ebxo4cstkdpdz1tZZ4PfOFDZwhvirE&export=download&confirm=t",
    archive, mode = "wb", quiet = TRUE
  )
}
actual_sha256 <- strsplit(system2("sha256sum", shQuote(archive), stdout = TRUE), " ")[[1]][1]
stopifnot(identical(actual_sha256, "ad351b23733c3aefefed4016bf84f64f7a77c074fe044c7a50cef153f283411e"))

unlink(pack_dir, recursive = TRUE, force = TRUE)
unzip(archive, exdir = session_root)
unlink(workspace, recursive = TRUE, force = TRUE)
dir.create(workspace, recursive = TRUE, showWarnings = FALSE)
status <- system2("python3", c(
  shQuote(file.path(pack_dir, "scripts", "materialize_case.py")),
  "--release", shQuote(pack_dir), "--task", task_id, "--workspace", shQuote(workspace)
))
stopifnot(status == 0)
setwd(workspace)
cat("Ready:", task_id, "\nInputs:", paste(list.files("inputs"), collapse = ", "), "\n")

## Explore the inputs

A short, task-relevant check of the data before you start.

In [ ]:
counts <- read.delim("inputs/counts.tsv", row.names = 1, check.names = FALSE)
clinical <- read.delim("inputs/clinical.tsv", check.names = FALSE)
cat(nrow(counts), "genes ×", ncol(counts), "samples",
    "· clinical rows:", nrow(clinical), "\n")
cat("First two genes:", paste(rownames(counts)[1:2], collapse = " and "), "\n")
plot(as.numeric(counts[1, ]), as.numeric(counts[2, ]),
     xlab = rownames(counts)[1], ylab = rownames(counts)[2],
     main = "Expression across matched samples", pch = 19)
print(utils::head(clinical, 4))

## Your solution

Edit this native-R cell and run it. It should read from `inputs/` and write the required artifact to `outputs/`.

In [ ]:
# Write your R solution below, then run this cell.
dir.create("outputs", showWarnings = FALSE)

# Inputs: inputs/clinical.tsv, inputs/counts.tsv
# Required output: outputs/outputs/correlation.txt
# Your code starts here:


In [ ]:
# @title Check output
score <- system2("python3", c(
  shQuote(file.path(pack_dir, "evaluator", "score_submission.py")),
  "--release", shQuote(pack_dir), "--task", task_id,
  "--outputs", shQuote(file.path(workspace, "outputs"))
), stdout = TRUE, stderr = TRUE)
raw_report <- paste(score, collapse = "\n")
report <- tryCatch(jsonlite::fromJSON(raw_report, simplifyVector = FALSE), error = function(e) NULL)
escape_html <- function(x) {
  x <- gsub("&", "&amp;", as.character(x), fixed = TRUE)
  x <- gsub("<", "&lt;", x, fixed = TRUE)
  x <- gsub(">", "&gt;", x, fixed = TRUE)
  x
}

if (is.null(report)) {
  IRdisplay::display_html(paste0(
    '<div style="border:1px solid #b42318;background:#fff4f2;padding:18px;border-radius:8px">',
    '<div style="font-size:20px;font-weight:700;color:#b42318">CHECK ERROR</div>',
    '<pre style="white-space:pre-wrap">', escape_html(raw_report), '</pre></div>'
  ))
} else if (isTRUE(report$pass)) {
  IRdisplay::display_html(paste0(
    '<div style="border:1px solid #1f7a46;background:#eefbf3;padding:18px;border-radius:8px">',
    '<div style="font-size:22px;font-weight:700;color:#176b3a">✓ PASS</div>',
    '<div style="margin-top:6px;color:#245b3c">All required artifacts match the reference checks.</div></div>'
  ))
} else {
  failures <- unlist(lapply(names(report$comparisons), function(file) {
    item <- report$comparisons[[file]]
    if (isTRUE(item$match)) return(NULL)
    reason <- if (is.null(item$reason)) "Output did not match the reference." else item$reason
    tier <- if (is.null(item$tier)) "" else paste0(' <span style="color:#777">[', escape_html(item$tier), ']</span>')
    paste0('<li style="margin:8px 0"><strong>', escape_html(file), '</strong>: ',
           escape_html(reason), tier, '</li>')
  }))
  IRdisplay::display_html(paste0(
    '<div style="border:1px solid #b42318;background:#fff4f2;padding:18px;border-radius:8px">',
    '<div style="font-size:22px;font-weight:700;color:#b42318">✕ FAIL</div>',
    '<div style="margin-top:6px;color:#7a271a">The output needs attention:</div>',
    '<ul style="margin:8px 0 0;padding-left:22px">', paste(failures, collapse = ""), '</ul></div>'
  ))
}

## Reference solution.R

If you are stuck or want to compare approaches, the exact frozen reference solution is below. Run **Setup** again before using it if you want a clean workspace.

In [ ]:
run_task <- function(work_dir) {
  counts_file <- file.path(work_dir, "inputs", "counts.tsv")
  counts <- read.table(counts_file, sep="\t", header=TRUE, row.names=1)
  
  gene1 <- as.numeric(counts[1, ])
  gene2 <- as.numeric(counts[2, ])
  
  cor_val <- cor(gene1, gene2, method="spearman")
  
  # Save output
  writeLines(as.character(cor_val), file.path(work_dir, "outputs", "correlation.txt"))
  
  return(cor_val)
}

run_task(getwd())